## <메트로폴리탄 미술관 데이터 수집>
- 흉배는 API 활용, 그 외 나머지는 직접 수집 예정
- API 설명
   - https://metmuseum.github.io/

### 1. 필요한 것들 불러오기

In [1]:
import os
import time
import requests
import pandas as pd
 
OUTPUT_EXCEL = "../data/met_hyungbae.xlsx"
HEADERS = {"User-Agent": "aks-digital-humanities-research/1.0"}

### 2. API 호출해서 원하는 정보 얻기

In [3]:
# 1. Museum
MUSEUM_CODE = "MET"
CATEGORY = "흉배"
CATEGORY_LETTER = "H"
KEYWORD = "rank badge"

rows = []
serial_counter = {}
failed_ids = []

def next_temp_id(museum_code):
    n = serial_counter.get(museum_code, 0) + 1
    serial_counter[museum_code] = n
    return f"Y{museum_code}{CATEGORY_LETTER}{n:02d}"

def get_with_retry(url, max_retries=4, base_wait=2):
    for attempt in range(max_retries):
        resp = requests.get(url, headers=HEADERS, timeout=20)
        if resp.status_code == 200:
            return resp
        if resp.status_code == 403:
            wait = base_wait * (attempt + 1)
            print(f"403 -> {wait}초 대기 후 재시도 ({attempt+1}/{max_retries})")
            time.sleep(wait)
            continue
        resp.raise_for_status()
    resp.raise_for_status()
    return resp

def fetch_met(keyword=KEYWORD):
    base = "https://collectionapi.metmuseum.org/public/collection/v1"
    r = requests.get(f"{base}/search", params={"q": keyword}, headers=HEADERS, timeout=20)
    r.raise_for_status()
    ids = r.json().get("objectIDs") or []
    for obj_id in ids:
        time.sleep(0.5)

        try:
            resp = requests.get(f"{base}/objects/{obj_id}", headers=HEADERS, timeout=20)
            resp.raise_for_status()
            detail = resp.json()
        except (requests.exceptions.RequestException, ValueError) as e:
            print(f"objectID {obj_id} 실패: {e}")
            failed_ids.append(obj_id)
            continue

        temp_id = next_temp_id(MUSEUM_CODE)

        all_images = [u for u in [detail.get("primaryImage")] + (detail.get("additionalImages") or []) if u]

        rows.append({
            # --- A~I: 마스터 스키마 ---
            "임시ID": temp_id,
            "분류": CATEGORY,
            "소장처": "메트로폴리탄 미술관",
            "소장처유물번호": f"Met {detail.get('accessionNumber')}",
            "한글명": "",
            "한자명": "",
            "영어명": detail.get("title"),
            "URL": detail.get("objectURL"),
            # --- 나머지: 있는 거 다 ---
            "searched_keyword": keyword,
            "culture": detail.get("culture"),
            "period": detail.get("period"),
            "date": detail.get("objectDate"),
            "medium": detail.get("medium"),
            "classification": detail.get("classification"),
            "dimensions": detail.get("dimensions"),
            "credit_line": detail.get("creditLine"),        # 추가
            "curatorial_department": detail.get("department"),  # 추가
            "image_url": detail.get("primaryImage"),
            "image_urls": "; ".join(all_images),
            "is_public_domain": detail.get("isPublicDomain"),
            "license_note": "Met Open Access (CC0 for public-domain works)",
        })

    if failed_ids:
        print(f"총 {len(failed_ids)}건 실패, 나중에 재시도 필요: {failed_ids}")

fetch_met()
print(f"{len(rows)}건 수집 완료")

objectID 72339 실패: 404 Client Error: Not Found for url: https://collectionapi.metmuseum.org/public/collection/v1/objects/72339
총 1건 실패, 나중에 재시도 필요: [72339]
257건 수집 완료


### 3. 데이터 프레임 -> 엑셀

In [4]:
df = pd.DataFrame(rows)
master_cols = ["임시ID", "분류", "소장처", "소장처유물번호", "한글명", "한자명", "영어명", 
               "URL", "culture","period","date","medium","classification","dimensions","credit_line",
               "curatorial_department","image_urls","is_public_domain","license_note"]
other_cols = [c for c in df.columns if c not in master_cols]
df = df[master_cols + other_cols]

os.makedirs("../data", exist_ok=True)

pure_hyungbae = df[df["영어명"].str.contains("Rank Badge", na=False)]
pure_hyungbae.to_excel(OUTPUT_EXCEL, index=False)

print(pure_hyungbae.shape)
pure_hyungbae.head()

(209, 21)


,임시ID,분류,소장처,소장처유물번호,한글명,한자명,영어명,URL,culture,period,...,medium,classification,dimensions,credit_line,curatorial_department,image_urls,is_public_domain,license_note,searched_keyword,image_url
0,YMETH01,흉배,메트로폴리탄 미술관,Met 30.75.968,,,Rank Badge,https://www.metmuseum.org/art/collection/searc...,China,Qing dynasty (1644–1911),...,Silk on silk,Textiles-Embroidered,Overall: 12 1/4 x 12 1/2 in. (31.1 x 31.8cm),"Bequest of William Christian Paul, 1929",Asian Art,https://images.metmuseum.org/CRDImages/as/orig...,True,Met Open Access (CC0 for public-domain works),rank badge,https://images.metmuseum.org/CRDImages/as/orig...
1,YMETH02,흉배,메트로폴리탄 미술관,Met 53.60.20,,,Rank Badge,https://www.metmuseum.org/art/collection/searc...,Korea,Joseon dynasty (1392–1910),...,Silk,Textiles-Embroidered,7 3/4 x 6 3/4 in. (19.7 x 17.1 cm),"Seymour Fund, 1953",Asian Art,https://images.metmuseum.org/CRDImages/as/orig...,True,Met Open Access (CC0 for public-domain works),rank badge,https://images.metmuseum.org/CRDImages/as/orig...
2,YMETH03,흉배,메트로폴리탄 미술관,Met 53.60.21,,,Rank Badge,https://www.metmuseum.org/art/collection/searc...,Korea,Joseon dynasty (1392–1910),...,Silk,Textiles-Embroidered,7 1/2 x 6 1/2 in. (19.1 x 16.5 cm),"Seymour Fund, 1953",Asian Art,https://images.metmuseum.org/CRDImages/as/orig...,True,Met Open Access (CC0 for public-domain works),rank badge,https://images.metmuseum.org/CRDImages/as/orig...
3,YMETH04,흉배,메트로폴리탄 미술관,Met 53.60.22,,,Rank Badge,https://www.metmuseum.org/art/collection/searc...,Korea,Joseon dynasty (1392–1910),...,Silk,Textiles-Embroidered,7 1/2 x 6 1/2 in. (19.1 x 16.5 cm),"Seymour Fund, 1953",Asian Art,https://images.metmuseum.org/CRDImages/as/orig...,True,Met Open Access (CC0 for public-domain works),rank badge,https://images.metmuseum.org/CRDImages/as/orig...
4,YMETH05,흉배,메트로폴리탄 미술관,Met 53.60.16,,,Rank Badge,https://www.metmuseum.org/art/collection/searc...,Korea,Joseon dynasty (1392–1910),...,Silk,Textiles-Embroidered,8 1/2 x 7 3/4 in. (21.6 x 19.7 cm),"Seymour Fund, 1953",Asian Art,https://images.metmuseum.org/CRDImages/as/orig...,True,Met Open Access (CC0 for public-domain works),rank badge,https://images.metmuseum.org/CRDImages/as/orig...


### 4. 이미지

In [20]:
# 이미지 저장 폴더 (NFM 노트북이랑 동일한 경로 구조)
os.makedirs("../image/hyungbae", exist_ok=True)

pure_hyungbae = pure_hyungbae.reset_index(drop=True)

for i, row in pure_hyungbae.iterrows():
    temp_id = row["임시ID"]
    raw = row["image_urls"]

    if pd.isna(raw) or str(raw).strip() == "":
        continue  # 이미지 없는 유물은 건너뜀

    image_urls = [u for u in str(raw).split("; ") if u.strip()]

    for idx, image_url in enumerate(image_urls):
        suffix = "" if len(image_urls) == 1 else f"-{idx + 1}"
        filepath = f"../image/hyungbae/{temp_id}{suffix}.jpg"

        resp = requests.get(image_url, headers=HEADERS, timeout=30)
        with open(filepath, "wb") as f:
            f.write(resp.content)

    if (i + 1) % 20 == 0:
        print(f"{i + 1} / {len(pure_hyungbae)} 완료")

    time.sleep(0.5)

20 / 209 완료
40 / 209 완료
60 / 209 완료
80 / 209 완료
100 / 209 완료
120 / 209 완료
140 / 209 완료
160 / 209 완료
180 / 209 완료
